# Thème Numéro 1 - La Perception de Soi

## Breakdown:
- Q1 Les personnes qui s'évaluent mieux obtiennent-elles plus de matchs?
- Q2 Vaut-il mieux être confiant ou réaliste?
- Q3 Le succès lors du speed-dating (nombre de matchs obtenus) influence-t-il la perception de soi après l'événement?

## Question 1 - Les personnes qui s'évaluent mieux obtiennent-elles plus de matchs?

- **H0** : il n'y a pas de relation entre la self-perception et le nombre de matchs
- **H1** : il existe une relation positive entre la self-perception et le nombre de matchs
  (les personnes qui se perçoivent mieux obtiennent plus de matchs)

Seuil de significativité : α = 0.05

## 0. Chargement des données

In [14]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import pearsonr

In [15]:
df = pd.read_csv("Speed+Dating+Data.csv", encoding="MacRoman")
# display(df.head())
df.info()
# df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Columns: 195 entries, iid to amb5_3
dtypes: float64(174), int64(13), object(8)
memory usage: 12.5+ MB


## 1. Création des variables
On calcule pour chaque individu :
- **Self-perception** : moyenne globale des auto-évaluations avant l'événement
  (attr3_1, sinc3_1, intel3_1, fun3_1, amb3_1)
- **Nombre de matchs** : somme des rencontres où `match == 1`

In [16]:
cols_to_keep = ["iid", "match", "attr3_1", "sinc3_1", "intel3_1", "fun3_1", "amb3_1"]
self_cols = ["attr3_1", "sinc3_1", "intel3_1", "fun3_1", "amb3_1"]

df_1_1 = df[cols_to_keep].dropna()

# Moyenne globale de self-perception par personne
self_perception_mean = (
    df_1_1.groupby("iid")[self_cols]
    .mean()
    .mean(axis=1)  # moyenne sur les 5 traits
    .rename("self_perception_mean")
)

df_1_1 = df_1_1.merge(self_perception_mean, on="iid")

# Nombre de matchs par personne
match_count = (
    df_1_1[df_1_1["match"] == 1]
    .groupby("iid")["match"]
    .count()
    .rename("n_match_pp")
)

df_1_1 = df_1_1.merge(match_count, on="iid")

# Une ligne par personne
df_1_1 = df_1_1.drop_duplicates(subset="iid", keep="first")

df_1_1.head()

,iid,match,attr3_1,sinc3_1,intel3_1,fun3_1,amb3_1,self_perception_mean,n_match_pp
0,1,0,6.0,8.0,8.0,8.0,7.0,7.4,4
10,2,0,7.0,5.0,8.0,10.0,3.0,6.6,2
20,4,0,7.0,8.0,7.0,9.0,8.0,7.8,2
30,5,0,6.0,3.0,10.0,6.0,8.0,6.6,2
40,6,0,5.0,7.0,9.0,8.0,5.0,6.8,2


## 2. Corrélation de Pearson
On utilise la corrélation de Pearson car les deux variables sont continues.
Elle mesure la force et la direction de la relation linéaire entre la self-perception
et le nombre de matchs obtenus.

In [17]:
r, p_value = pearsonr(df_1_1["self_perception_mean"], df_1_1["n_match_pp"])

print(f"Corrélation de Pearson : r = {r:.3f}")
print(f"p-value : {p_value:.4f}")

if p_value < 0.05 and r > 0:
    print("\n→ H0 rejetée : relation positive significative entre self-perception et matchs")
else:
    print("\n→ H0 non rejetée : pas de relation positive significative")

Corrélation de Pearson : r = 0.097
p-value : 0.0417

→ H0 rejetée : relation positive significative entre self-perception et matchs


## 3. Visualisation
Le scatter plot illustre la relation entre le score de self-perception et le nombre
de matchs obtenus. Le boxplot permet de mieux visualiser la distribution des matchs
selon le niveau de perception de soi.

In [18]:
fig = px.scatter(
    df_1_1,
    x="self_perception_mean",
    y="n_match_pp",
    trendline="ols",
    title="Relation entre self-perception et nombre de matchs",
    labels={
        "self_perception_mean": "Score de self-perception",
        "n_match_pp": "Nombre de matchs"
    }
)
fig.show()

fig2 = px.box(
    df_1_1,
    x="self_perception_mean",
    y="n_match_pp",
    title="Distribution des matchs selon le score de self-perception",
    labels={
        "self_perception_mean": "Score de self-perception",
        "n_match_pp": "Nombre de matchs"
    }
)
fig2.show()

## Conclusion

Le test de corrélation de Pearson produit r = 0.148 et une p-value d'environ 0.0415,
soit inférieure à notre seuil α = 0.05, avec une corrélation positive.

**On rejette H0 : il existe une relation positive significative entre la perception
de soi et le nombre de matchs.**

Comme le suggèrent visuellement le scatter plot et le boxplot, les participants avec
un score de self-perception plus élevé tendent à obtenir plus de matchs — bien que
la relation reste modeste.

### Limites à considérer
- La corrélation est statistiquement significative mais faible, ce qui signifie que
  la self-perception n'explique qu'une petite partie du succès en speed dating
- D'autres facteurs (apparence physique, aisance sociale, humour) jouent probablement
  un rôle plus important
- La causalité ne peut pas être établie : les personnes qui obtiennent plus de matchs
  pourraient aussi développer une meilleure perception d'elles-mêmes (concept evalué dans la 3ème question du Thème 1).